# MagicKey
> Passwordless auth combining magic links and passkeys

Passwords are a hassle — users forget them, reuse them, and leak them. Besides [OAuth](./oauth.html) two other alternatives have become popular: 

1. **magic links**: one-time URLs sent to your email
2.  **passkeys**: cryptographic credentials stored on your device that (optionally) authenticate via biometrics or PIN

Each is great on its own, but combining them gives you the best of both worlds: magic links handle sign-up and device recovery, while passkeys make returning logins instant and phishing-resistant. MagicKey implements this combination for FastHTML apps. 

On this page you'll see how to set up MagicKey and walk through the full authentication flow.

## A Complete Example

Like `OAuth`, you subclass `MagicKey` and override a few methods to connect it to your storage. You need to provide:

- **One `send_email` callable** that receives `(email, url)` and delivers the magic link
- **Two routes** — `/login` (with a passkey button and email form) and `/setup_passkey` (offering passkey registration after first magic link login)
- **Five storage methods** that map to however you store users and credentials: `get_user_id`, `has_passkey`, `get_passkey`, `save_passkey`, `update_passkey`

MagicKey handles everything else: beforeware for auth checks, token generation/verification, and all the WebAuthn ceremony routes. Here's a minimal app with an in-memory SQLite database:

In [ ]:
from fasthtml.common import *
from fasthtml.magickey import MagicKey

db = database(':memory:')
users = db.t.user.create(id=int, email=str, pk='id')
passkeys = db.t.passkey.create(id=str, user_id=int, public_key=bytes, sign_count=int, pk='id')
User,Passkey = users.dataclass(),passkeys.dataclass()

class Auth(MagicKey):
    def get_user_id(self, email):
        res = users(where="email = ?", where_args=[email])
        if res: return res[0].id
        return users.insert(User(email=email)).id
    def has_passkey(self, email):
        return bool(passkeys(where="user_id = ?", where_args=[self.get_user_id(email)]))
    def get_passkey(self, cred_id):
        try: r = passkeys[cred_id]
        except NotFoundError: return None
        return dict(email=users[r.user_id].email, public_key=r.public_key, sign_count=r.sign_count)
    def save_passkey(self, cred_id, email, public_key, sign_count):
        uid = self.get_user_id(email)
        passkeys.insert(Passkey(id=cred_id, user_id=uid, public_key=public_key, sign_count=sign_count))
    def update_passkey(self, cred_id, sign_count):
        passkeys.update(Passkey(id=cred_id, sign_count=sign_count))


Next we create the app and wire up MagicKey. The `send_email` callable receives an email and a magic link URL. In production you'd send a real email, here we just display the link for testing:

In [ ]:
app, rt = fast_app()
def fake_send_email(email, url): return P(f'Magic link for {email}: ', A(url, href=url))
mk = Auth(app, send_email=fake_send_email)

When you run `mk = Auth(app, send_email=fake_send_email)`, MagicKey adds beforeware that redirects unauthenticated users to `/login`, and registers all the backend routes for magic link and passkey handling. You can also pass `rp_name` (the app name shown in the browser's passkey prompt — defaults to the hostname) and `token_expiry` (magic link lifetime in seconds, default 3600).

The `/login` page wires up two entry points: a "Sign in with Passkey" button that triggers an htmx POST to `/request_passkey_auth`, and an email form that POSTs to `/send_magic_link`. The `/setup_passkey` page similarly triggers `/request_passkey_reg` for registration or `/skip_passkey_reg` to skip.

In [ ]:
@rt('/')
def home(auth):
    u = users[auth]
    return P(f'Hello {u.email}!'), A('Log out', href='/logout')

@rt('/login')
def login(error: str=None):
    errmsg = P(error.replace('_', ' ').title(), style='color:red') if error else ''
    return Titled('Sign In', errmsg,
                Button('Sign in with Passkey', hx_post='/request_passkey_auth', target_id='scripts'),
                Hr(),
                Form(method='post', action='/send_magic_link')(
                    Input(name='email', type='email', placeholder='you@example.com'),
                    Button('Send Magic Link')),
                Div(id='scripts'))

@rt('/setup_passkey')
def setup_passkey(): return Titled('Set Up Passkey',
                            P('Set up a passkey for faster logins next time?'),
                            Button('Register Passkey', hx_post='/request_passkey_reg', target_id='scripts'),
                            Form(Button('Skip'), method='post', action='/skip_passkey_reg'),
                            Div(id='scripts'))

## The Authentication Flow

Here's how the three main scenarios play out:

**First visit (magic link → passkey setup):**

A new user visits `/` and gets redirected to `/login` by the beforeware. They enter their email — this POSTs to `/send_magic_link`, which generates a one-time token (valid for `token_expiry` seconds, default 1 hour) and calls your `send_email` with the link. When they click it, `/verify_magiclink` checks and marks the token as used, creates the user if new, and calls `after_magiclink_verify`. Since this user has no passkey yet, they're redirected to `/setup_passkey`, where the browser's WebAuthn API prompts for biometric/PIN registration. Done — they're in.

**Returning visit (passkey):**

The user clicks "Sign in with Passkey" on `/login`, which POSTs to `/request_passkey_auth`. The browser prompts for biometric/PIN, and the credential is verified server-side via `/verify_passkey_auth`. No email, no waiting — instant login.

**Returning visit (magic link fallback):**

If the passkey doesn't work (new device, declined prompt, etc.), the user falls back to the email form. Since they already have a passkey registered, `after_magiclink_verify` sees this and logs them in directly — no passkey setup prompt this time.

You can customise two aspects of this flow by overriding methods on your subclass:

1. `get_auth(user_id, session)` controls where users land after successful authentication (defaults to `/`). 
2. `after_magiclink_verify(email, session, req)` controls what happens after a magic link is verified — the default checks whether the user already has a passkey and either logs them in directly or redirects to `/setup_passkey`.

## Sending Magic Links

In the example above we provided a fake example that just prints the email. To actually send emails you can use any of the many email libraries available in Python.

Here's one example available to apps deployed on [Pla.sh](https://pla.sh), a python web app platform by the creators of FastHTML.

```py
from plash_cli.auth import send_magiclink
def send_email(email,url):
    res = send_magiclink(email,url)
    if res.status_code != 200: return P('Something went wrong, please try again later')
    else: return P('Please check your email for your login link')
mk = Auth(app, send_email=send_email)
...
```